# Demo 3 — Practical Notebook: Managing the Context Window (I / E / D)

This notebook walks through the practical steps from the lesson:

1) **Budget** the context window (Instructions / Example / Data)
2) **Order & delimit**: INSTRUCTIONS → EXAMPLE → CONTEXT
3) **Trim first**: remove repeats, side-quests, metadata
4) **Summarize** neutrally when still over budget
5) **Slot precisely**: 3–5 verbatim lines with IDs `[S1…]`
6) **Output contract**: one format, word cap, `sources`
7) **One-rule verifier**: pass/fail gate
8) **Decision rule**: summarize vs. retrieve
9) **Mini-exercise** to practice the workflow


In [1]:
# --- Setup: lightweight helpers (no external libraries) ---
from typing import List, Dict

def rough_token_count(text: str) -> int:
    """A quick, model-agnostic approximation (~4 chars/token baseline)."""
    if not text:
        return 0
    # heuristic: 1 token per 4 characters, min 1 per word
    chars = len(text)
    words = max(1, len(text.split()))
    return max(words, int(round(chars / 4)))

def budget_counts(i: str, e: str, d: str, cap: int = 800) -> Dict[str, int]:
    return {
        "cap": cap,
        "I": rough_token_count(i),
        "E": rough_token_count(e),
        "D": rough_token_count(d),
        "total": rough_token_count(i)+rough_token_count(e)+rough_token_count(d)
    }

def print_budget(b: Dict[str,int]):
    print(f"Cap: {b['cap']} | I: {b['I']} | E: {b['E']} | D: {b['D']} | Total: {b['total']} | {'OK' if b['total']<=b['cap'] else 'OVER'}\n")


## 1) Context Window as a Budget (I / E / D)
Adjust the cap and the split as needed. Draft first, then trim to fit.

In [2]:
cap = 800
I = "Return JSON with fields {answer, sources}. Keep answer ≤150 words. Use only provided context. If info is missing, set answer='' and sources=[]."
E = '{"answer": "...", "sources": ["S1"]}'
D = """[S1] WAU held at 24/28 for four weeks (85.7%); median session 11m40s.
[S2] Transcription latency improved from 7.1s to 4.3s via batching.
[S3] Marketing name change not approved; use 'Atlas Notes'."""
b = budget_counts(I, E, D, cap)
print_budget(b)
print(I)

Cap: 800 | I: 36 | E: 9 | D: 49 | Total: 94 | OK

Return JSON with fields {answer, sources}. Keep answer ≤150 words. Use only provided context. If info is missing, set answer='' and sources=[].


## 2) Order & Delimiters
Always order your sections and label them clearly. Fence long context so it isn’t mistaken for rules.

In [3]:
prompt = f"""
INSTRUCTIONS:
{I}

EXAMPLE:
{E}

CONTEXT:
```
{D}
```
"""
print(prompt)


INSTRUCTIONS:
Return JSON with fields {answer, sources}. Keep answer ≤150 words. Use only provided context. If info is missing, set answer='' and sources=[].

EXAMPLE:
{"answer": "...", "sources": ["S1"]}

CONTEXT:
```
[S1] WAU held at 24/28 for four weeks (85.7%); median session 11m40s.
[S2] Transcription latency improved from 7.1s to 4.3s via batching.
[S3] Marketing name change not approved; use 'Atlas Notes'.
```



## 3) Trim First: Three Quick Cuts
**Goal:** short, factual, cite-able context. We’ll simulate trimming the background used in class.

In [4]:
background = (
    "Pilot with 28 CSMs concluded last Friday. WAU held at 24/28 for 4 weeks (85.7%); "
    "median session 11m40s. Latency improved from 7.1s to 4.3s via batching. "
    "16 tickets (mostly how-to); 4 UI bugs fixed in 0.9.3. Security reviewed OAuth scopes; "
    "no blockers. Several asked for dark mode; calendar color-coding out of Q4 scope. "
    "Avoid pricing in this update. Deadline: Friday 5 PM ET. Video link and Jira refs omitted."
)

def quick_trim(text: str) -> str:
    # Simulate cuts: drop video link/Jira, marketing fluff marker words, keep decision-relevant facts
    lines = [s.strip() for s in text.split('.') if s.strip()]
    kept = []
    for s in lines:
        lower = s.lower()
        if any(k in lower for k in ["video", "jira", "marketing", "fluff", "self-reported"]):
            continue
        if any(k in lower for k in ["wau", "latency", "deadline", "security", "tickets", "bugs", "scope", "q4"]):
            kept.append(s)
    return '. '.join(kept) + '.'

trimmed = quick_trim(background)
print(trimmed)
print("\nRough tokens:", rough_token_count(trimmed))

WAU held at 24/28 for 4 weeks (85. Latency improved from 7. 16 tickets (mostly how-to); 4 UI bugs fixed in 0. Security reviewed OAuth scopes; no blockers. Several asked for dark mode; calendar color-coding out of Q4 scope. Deadline: Friday 5 PM ET.

Rough tokens: 62


## 4) Neutral, Task-Oriented Summary
Use 1–2 sentences: who/what/constraints. No new info, no opinions.

In [5]:
summary = (
    "Pilot shows steady use and improved transcription latency, with low-severity support volume and no security blockers. "
    "Provide a ≤150-word exec summary by Friday 5 PM ET; avoid pricing and keep naming consistent with 'Atlas Notes'."
)
print(summary)
print("Rough tokens:", rough_token_count(summary))

Pilot shows steady use and improved transcription latency, with low-severity support volume and no security blockers. Provide a ≤150-word exec summary by Friday 5 PM ET; avoid pricing and keep naming consistent with 'Atlas Notes'.
Rough tokens: 58


## 5) Slot Precisely (Tiny, Labeled Snippets)
Keep 3–5 verbatim lines you will cite. Give each a stable ID.

In [6]:
S1 = "Average weekly active users (WAU) held at 24 of 28 for four consecutive weeks (85.7%); median session length was 11m 40s."
S2 = "Regarding performance, we cut average transcription latency from 7.1s to 4.3s by batching requests."
S3 = "Delivery deadline is Friday 5 PM Eastern; summary goes into the weekly executive brief."
snippets = {"S1": S1, "S2": S2, "S3": S3}
for k,v in snippets.items():
    print(f"[{k}] {v}")
print("\nTotal snippet tokens:", sum(rough_token_count(v) for v in snippets.values()))

[S1] Average weekly active users (WAU) held at 24 of 28 for four consecutive weeks (85.7%); median session length was 11m 40s.
[S2] Regarding performance, we cut average transcription latency from 7.1s to 4.3s by batching requests.
[S3] Delivery deadline is Friday 5 PM Eastern; summary goes into the weekly executive brief.

Total snippet tokens: 77


## 6) Output Contract (One Format, Word Cap, Sources)
We’ll ask for JSON with fields `answer` and `sources`.

In [7]:
contract = (
    "Return JSON with fields: answer (≤150 words), sources (array of ['S*']). "
    "Use only provided CONTEXT. If info is missing, set answer='' and sources=[]."
)
print(contract)

Return JSON with fields: answer (≤150 words), sources (array of ['S*']). Use only provided CONTEXT. If info is missing, set answer='' and sources=[].


## 7) One-Rule Verifier (Pass/Fail)
A small gate that checks: format, word cap, required fields, and valid snippet references.

In [8]:
import json
def word_count(s: str) -> int:
    return len([w for w in s.split() if w])

def verify(answer_json: str, snippet_ids: List[str], max_words: int = 150) -> Dict[str, str]:
    try:
        obj = json.loads(answer_json)
    except Exception as e:
        return {"status": "FAIL", "reason": f"Invalid JSON: {e}"}
    if not isinstance(obj, dict) or "answer" not in obj or "sources" not in obj:
        return {"status": "FAIL", "reason": "Missing required fields (answer, sources)."}
    if word_count(obj["answer"]) > max_words:
        return {"status": "FAIL", "reason": f"Answer exceeds {max_words} words."}
    if not isinstance(obj["sources"], list) or any(s not in snippet_ids for s in obj["sources"]):
        return {"status": "FAIL", "reason": "Sources must reference existing snippet IDs."}
    return {"status": "PASS", "reason": "All checks passed."}

# Example (compliant)
example_output = json.dumps({
    "answer": (
        "Pilot outcome: steady use and improved latency. Adoption stayed at 24/28 WAU for four weeks (85.7%) and sessions were 11m40s (S1). "
        "Latency improved from 7.1s to 4.3s via batching (S2). Submit ≤150-word summary by Friday 5 PM ET (S3)."
    ),
    "sources": ["S1","S2","S3"]
})
print(example_output)
verify(example_output, list(snippets.keys()))

{"answer": "Pilot outcome: steady use and improved latency. Adoption stayed at 24/28 WAU for four weeks (85.7%) and sessions were 11m40s (S1). Latency improved from 7.1s to 4.3s via batching (S2). Submit \u2264150-word summary by Friday 5 PM ET (S3).", "sources": ["S1", "S2", "S3"]}


{'status': 'PASS', 'reason': 'All checks passed.'}

## 8) Decision Rule: Summarize vs. Retrieve
If you have all needed facts and can stay within budget, **summarize**; otherwise, **retrieve** (next module).

In [9]:
def decide_summarize_vs_retrieve(have_all_facts: bool, estimated_tokens: int, cap: int = 800) -> str:
    if have_all_facts and estimated_tokens <= cap:
        return "Summarize"
    return "Retrieve (use RAG in next module)"

decision = decide_summarize_vs_retrieve(have_all_facts=True, estimated_tokens=b["total"], cap=cap)
decision

'Summarize'

## 9) Mini-Exercise (Hands-on)
**Instructions:**
1. Trim the background to ≤100 words (delete repeats/side-quests/metadata).
2. Write a neutral, 2-sentence summary (who, what, constraints).
3. Slot 3–5 verbatim lines with IDs `[S1…S5]`.
4. Produce JSON with `answer` (≤150 words) and `sources` referencing your IDs.


In [10]:
exercise_background = (
    "Background (project update v2): Atlas Notes pilot with 28 CSMs; WAU 24/28 for 4 weeks (85.7%); median 11m40s. "
    "Latency improved 7.1s→4.3s via batching. 16 tickets; 4 UI bugs fixed in 0.9.3; others how-to. Security review: no blockers. "
    "Avoid pricing; naming is 'Atlas Notes' (marketing alt not approved). Deadline Friday 5 PM ET for ≤150-word exec summary."
)
print(exercise_background)

# TODO: Learner implements their own quick trimming here
learner_trimmed = ""  # <-- fill in up to 100 words
learner_summary = ""   # <-- two sentences
S1 = ""; S2 = ""; S3 = ""  # <-- verbatim snippets
learner_snippets = {k:v for k,v in {"S1":S1, "S2":S2, "S3":S3}.items() if v}

learner_output = json.dumps({"answer": "", "sources": list(learner_snippets.keys())})
print(learner_output)

# Run verifier when ready
# verify(learner_output, list(learner_snippets.keys()))

Background (project update v2): Atlas Notes pilot with 28 CSMs; WAU 24/28 for 4 weeks (85.7%); median 11m40s. Latency improved 7.1s→4.3s via batching. 16 tickets; 4 UI bugs fixed in 0.9.3; others how-to. Security review: no blockers. Avoid pricing; naming is 'Atlas Notes' (marketing alt not approved). Deadline Friday 5 PM ET for ≤150-word exec summary.
{"answer": "", "sources": []}
